# 02 — Build and score a submission

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jean-jsj/CARD/blob/main/examples/02_score_a_submission.ipynb)

This notebook runs the whole evaluation loop once, end to end, on the mini slice:

1. build the three submission CSVs from a deliberately naive model,
2. score them against the cell's hidden truth with the standard harness,
3. read the score report — the same numbers the leaderboard uses.

The naive model here is the floor: forecast = recent average, elasticities = 0,
counterfactual demand changes = 0 ("prices don't matter"). You should beat it
easily, and the point of the exercise is that afterwards you know exactly what
files to produce and what report you get back.

In [1]:
# Setup: works from a local clone or on Colab (clones the repo, no pip install needed).
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("metrics").is_dir():
    subprocess.run(["git", "clone", "-q", "https://github.com/jean-jsj/CARD"], check=True)
    os.chdir("CARD")
elif Path.cwd().name == "examples":
    os.chdir("..")
assert Path("metrics").is_dir(), "run this notebook from the CARD repo root or on Colab"
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 11, "font.size": 10})
BLUE, GREEN, GRAY = "#2a78d6", "#008300", "#6b7280"

In [2]:
# Fetch the ~18 MB starter slice of the endogeneity-on cell (10 stores, all 40 products).
from huggingface_hub import snapshot_download

CELL = "complex_log_log_endogenous_seed001"
cell_dir = Path("benchmark/dev_mini") / CELL
if not cell_dir.exists():
    snapshot_download(repo_id="jean-jsj/CARD", repo_type="dataset",
                      allow_patterns=[f"dev_mini/{CELL}/*"], local_dir="benchmark")
print("data at:", cell_dir)

data at: benchmark/dev_mini/complex_log_log_endogenous_seed001


## 1. Build the three submission files

A submission for one cell is a directory with up to three CSVs. Only `public/`
files go into the model — `hidden/` exists for scoring and is never model input.

In [3]:
sub_dir = Path("submissions_local/naive_mini") / CELL
sub_dir.mkdir(parents=True, exist_ok=True)

train    = pd.read_csv(cell_dir / "public" / "transactions_train_public.csv")
holdout  = pd.read_csv(cell_dir / "public" / "transactions_holdout_context_public.csv")
products = pd.read_csv(cell_dir / "public" / "products_public.csv")
sweep    = pd.read_csv(cell_dir / "public" / "counterfactual_sweep_context_public.csv")

# forecast_predictions.csv - mean units per (product, store) over the last 8 training weeks.
recent = train[train.week > train.week.max() - 8]
mean_units = recent.groupby(["product_id", "store_id"])["units"].mean().rename("predicted_units")
forecast = holdout[["product_id", "store_id", "week"]].merge(
    mean_units, on=["product_id", "store_id"], how="left").fillna({"predicted_units": 0.0})
forecast.to_csv(sub_dir / "forecast_predictions.csv", index=False)

# elasticity_matrix.csv - the all-zeros 40x40 matrix (the no-information value).
from itertools import product as iproduct
ids = products.product_id.tolist()
pd.DataFrame([(j, i, 0.0) for j, i in iproduct(ids, ids)],
             columns=["priced_product_id", "affected_product_id", "elasticity"]
             ).to_csv(sub_dir / "elasticity_matrix.csv", index=False)

# counterfactual_deltas.csv - zero demand change everywhere ("prices don't matter").
deltas = sweep[["intervention_id", "product_id", "store_id", "week"]].drop_duplicates()
deltas.assign(predicted_delta_units=0.0).to_csv(sub_dir / "counterfactual_deltas.csv", index=False)

print("wrote:", *[p.name for p in sorted(sub_dir.iterdir())], sep="\n  ")

wrote:
  counterfactual_deltas.csv
  elasticity_matrix.csv
  forecast_predictions.csv


## 2. Score it

`metrics.evaluate_submission` grades one submission directory against one cell
directory and writes a JSON report. On the dev seed this is local and instant —
the truth ships with the cell.

In [4]:
import json, subprocess

score_path = Path("submissions_local/naive_mini_score.json")
subprocess.run([sys.executable, "-m", "metrics.evaluate_submission",
                "--cell-dir", str(cell_dir),
                "--submission-dir", str(sub_dir),
                "--submission-name", "naive_mini",
                "--out", str(score_path)], check=True)
score = json.loads(score_path.read_text())
print("scored tasks:", [k for k in ("sales_forecasting", "elasticity_recovery",
                                    "counterfactual_prediction") if k in score])

wrote submissions_local/naive_mini_score.json
scored tasks: ['sales_forecasting', 'elasticity_recovery', 'counterfactual_prediction']


## 3. Read the report

**The headline (ranked): own-price bias.** The signed error of the predicted
demand change for the flagship product under its +10% scenario. 0 = unbiased; the
sign shows over- vs under-shoot. The naive model predicts *no* change, so its
error is the entire true response: bias = +1.0 (it understates the demand
response completely).

In [5]:
cf = score["counterfactual_prediction"]["headline"]
print("scenario:          ", cf["scenario"])
print("own-price bias:    ", round(cf["own_price"]["own_price_wmpe"], 3), " (0 = unbiased)")
print("substitution error:", round(cf["substitution"]["substitution_wape"], 3), " (1.0 = predicted no change)")

scenario:           sweep_single_share_highest_plus10
own-price bias:     1.0  (0 = unbiased)
substitution error: 1.0  (1.0 = predicted no change)


**Displayed beside it (never ranked): the sales forecast error** — a
revenue-weighted WMAPE over the 16 withheld weeks, the number a conventional
forecasting leaderboard would rank by. CARD displays the two side by side because
its central claim is that they diverge: a model can win on forecast error and
still carry a large own-price bias.

In [6]:
l1 = score["sales_forecasting"]
print("forecast error (WMAPE):", round(l1["demand_wmape"], 3), " lower = better fit")
print("forecast bias  (WMPE): ", round(l1["demand_wmpe"], 3))

forecast error (WMAPE): 0.544  lower = better fit
forecast bias  (WMPE):  -0.017


**Elasticity recovery** localizes *which* responses a method gets wrong: sign
accuracy, substitute/complement F1, cross-effect ranking (NDCG), magnitude error,
and bias — for the diagonal (own-price) and the off-diagonal (cross-price)
separately.

In [7]:
el = score["elasticity_recovery"]
print("own-price sign accuracy:", el["own_price"]["sign_accuracy"], " (all-zeros scores 0)")
print("own-price magnitude WMAPE:", el["own_price"]["wmape"])
print("cross-price WMAPE:", el["cross_price"]["all_pairs"]["wmape"])

own-price sign accuracy: 0.0  (all-zeros scores 0)
own-price magnitude WMAPE: 1.0
cross-price WMAPE: 1.0


## Where you stand

Full-cell numbers for the four reference models (each fits the cell's own demand
system from public files, crossing instruments x text) are browsable in
[docs/reference_results.html](../docs/reference_results.html) and live in
[`submissions/`](../submissions/). On the arena cell the instrumented corners sit
near own-price bias 0, the naive corner near -0.23, and your all-zeros floor at
+1.0.

Two things to keep in mind when comparing:

- Mini-slice numbers are for orientation — leaderboard entries are scored on the
  **full** cells (`examples/download_data.py` without `--mini`).
- Dev-seed numbers are self-reported; the maintainer verifies leaderboard entries
  on private eval seeds ([CONTRIBUTING.md](../CONTRIBUTING.md)).

## Next

**[03_endogeneity_and_instruments.ipynb](03_endogeneity_and_instruments.ipynb)**
shows the benchmark's core phenomenon live: the naive estimator's elasticities
shift when the endogeneity knob turns on, and the released instrument removes the
shift.